# 06 – Lévy Increment Recovery

Recovers the latent Lévy increments dL^Y (temperature) and dL^X (log-price)
from the Kalman-filtered state estimates using the **exact discrete propagator**
formula derived from the Van Loan discretisation.

### Key fixes vs. original
- **CE-3 fixed**: CSV written with `mode="w"` (was `mode="a"`, causing duplicates).
- **QR-4 fixed**: increments recovered for **both** temperature **and** log-price.
- **LI-3 fixed**: uses MLE parameters (year⁻¹) and `dt = DT_YEARS`; no hard-coded λ.
- Exact formula replaces trapezoidal approximation.

Inputs:  `results/carma_temp_mle.json`, `results/carma_price_mle.json`,
         `results/x_hat_temp.npy`, `results/x_hat_price.npy`
Outputs: `data/increments/temp_inc.csv`, `data/increments/price_inc.csv`,
         `figures/increment_diagnostics.png`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

from config import (
    RES_DIR, FIG_DIR, INCR_DIR, DT_YEARS,
    TEMP_RESID_TRAIN_CSV, PRICE_RESID_TRAIN_CSV,
)
from carma_utils import build_companion, recover_increments_exact, load_params

# ── Load MLE parameters ───────────────────────────────────────────────────────
params_temp  = load_params(RES_DIR / 'carma_temp_mle.json')
params_price = load_params(RES_DIR / 'carma_price_mle.json')

print('Temperature MLE params:')
print(f'  a_coeffs : {np.round(params_temp["a_coeffs"], 5)}')
print(f'  b_coeffs : {np.round(params_temp["b_coeffs"], 5)}')
print(f'  sigma    : {params_temp["sigma"]:.6f}  yr^(-1/2)')
print(f'  Re(eigs) : {np.round(params_temp["eigenvalues_real"], 4)}')

print('\nLog-price MLE params:')
print(f'  a_coeffs : {np.round(params_price["a_coeffs"], 5)}')
print(f'  b_coeffs : {np.round(params_price["b_coeffs"], 5)}')
print(f'  sigma    : {params_price["sigma"]:.6f}  yr^(-1/2)')
print(f'  Re(eigs) : {np.round(params_price["eigenvalues_real"], 4)}')

# ── Load Kalman-filtered states ───────────────────────────────────────────────
x_hat_temp  = np.load(RES_DIR / 'x_hat_temp.npy')   # (N_temp, p_t)
x_hat_price = np.load(RES_DIR / 'x_hat_price.npy')  # (N_price, p_p)

print(f'\nFiltered states:')
print(f'  x_hat_temp  shape: {x_hat_temp.shape}')
print(f'  x_hat_price shape: {x_hat_price.shape}')

## 1.  Recover increments via exact propagator

The exact discrete-time update is:
$$Z_{t+\Delta} = F\,Z_t + B_\text{load}\,\Delta L_t, \qquad F = e^{A\Delta t}$$

Solving for the scalar increment:
$$\Delta L_t = \frac{B_\text{load}^\top (Z_{t+\Delta} - F\,Z_t)}{\|B_\text{load}\|^2},
\quad B_\text{load} = A^{-1}(F - I)\,B$$

This replaces the original trapezoidal approximation.

In [ ]:
def build_B(sigma, p):
    """Noise loading vector B = sigma * e_p (last basis vector)."""
    B = np.zeros((p, 1))
    B[-1, 0] = sigma
    return B

# ── Temperature increments ────────────────────────────────────────────────────
a_t = np.array(params_temp['a_coeffs'])
s_t = float(params_temp['sigma'])
p_t = len(a_t)
A_temp = build_companion(a_t)
B_temp = build_B(s_t, p_t)

dL_temp = recover_increments_exact(A_temp, B_temp, x_hat_temp, DT_YEARS)

print(f'Temperature increments: {len(dL_temp):,} values')
print(f'  mean={dL_temp.mean():.6f}  std={dL_temp.std():.6f}')
print(f'  skew={float(pd.Series(dL_temp).skew()):.4f}  '
      f'kurt={float(pd.Series(dL_temp).kurt()):.4f}  '
      f'(excess kurtosis)')
print(f'  min={dL_temp.min():.4f}  max={dL_temp.max():.4f}')

# ── Log-price increments ──────────────────────────────────────────────────────
a_p = np.array(params_price['a_coeffs'])
s_p = float(params_price['sigma'])
p_p = len(a_p)
A_price = build_companion(a_p)
B_price = build_B(s_p, p_p)

dL_price = recover_increments_exact(A_price, B_price, x_hat_price, DT_YEARS)

print(f'\nLog-price increments: {len(dL_price):,} values')
print(f'  mean={dL_price.mean():.6f}  std={dL_price.std():.6f}')
print(f'  skew={float(pd.Series(dL_price).skew()):.4f}  '
      f'kurt={float(pd.Series(dL_price).kurt()):.4f}  '
      f'(excess kurtosis)')
print(f'  min={dL_price.min():.4f}  max={dL_price.max():.4f}')

## 2.  Diagnostic plots: ACF and histograms

Recovered increments should be (approximately) i.i.d. if the CARMA model
is correctly specified. We check:
- **ACF** of increments and squared increments (serial independence)
- **Ljung-Box** whiteness test at lags 24, 48, 72
- **Histogram** vs Gaussian reference

In [ ]:
from scipy import stats as scipy_stats

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
xg = np.linspace(-5, 5, 300)

for row, (label, dL) in enumerate([
        ('Temperature', dL_temp),
        ('Log-price',   dL_price),
    ]):
    dL_std = (dL - dL.mean()) / dL.std()

    # Whiteness tests
    lb = acorr_ljungbox(dL, lags=[24, 48, 72], return_df=True)
    lb2 = acorr_ljungbox(dL**2, lags=[24], return_df=True)
    jb = scipy_stats.jarque_bera(dL)
    print(f'{label} increments:')
    print(f'  Ljung-Box (raw)   lag24 p={lb["lb_pvalue"].iloc[0]:.4f}  '
          f'lag48 p={lb["lb_pvalue"].iloc[1]:.4f}  lag72 p={lb["lb_pvalue"].iloc[2]:.4f}')
    print(f'  Ljung-Box (sq)    lag24 p={lb2["lb_pvalue"].iloc[0]:.4f}')
    print(f'  Jarque-Bera       stat={jb[0]:.1f}  p={jb[1]:.2e}')
    tag = 'PASS' if lb['lb_pvalue'].iloc[0] > 0.05 else 'FAIL'
    print(f'  Whiteness lag24: {tag}\n')

    # ACF of increments
    ax = axes[row, 0]
    plot_acf(dL, lags=72, ax=ax, markersize=2)
    ax.set_title(f'{label}: ACF of increments')

    # ACF of squared increments (ARCH check)
    ax = axes[row, 1]
    plot_acf(dL**2, lags=72, ax=ax, markersize=2)
    ax.set_title(f'{label}: ACF of squared increments')

    # Histogram vs Gaussian
    ax = axes[row, 2]
    ax.hist(dL_std, bins=200, density=True, alpha=0.7, label='empirical')
    ax.plot(xg, scipy_stats.norm.pdf(xg), 'r-', lw=2, label='N(0,1)')
    ax.set_xlim(-8, 8)
    ax.set_title(f'{label}: standardised increments')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'increment_diagnostics.png', dpi=150)
plt.show()

## 3.  Cross-correlation between temperature and log-price increments

The temperature and log-price series overlap only for 2023–2024 (price training
window). We align on the shorter series and compute the empirical cross-correlation
function at lags −72 … +72 h. Non-zero CCF at lag 0 motivates the coupling term Γ
in the joint CARMA model (notebook 08).

In [ ]:
# Align on the shorter (price) series length
n_min = min(len(dL_temp), len(dL_price))
dL_t_aligned = dL_temp[-n_min:]   # take the tail (most recent, same period)
dL_p_aligned = dL_price[-n_min:]

# Normalise to zero mean, unit variance
dL_t_n = (dL_t_aligned - dL_t_aligned.mean()) / dL_t_aligned.std()
dL_p_n = (dL_p_aligned - dL_p_aligned.mean()) / dL_p_aligned.std()

# Cross-correlation at lags -max_lag … +max_lag
max_lag = 72
lags = np.arange(-max_lag, max_lag + 1)
ccf  = np.array([
    np.corrcoef(dL_t_n[:n_min - abs(k)], dL_p_n[max(0, k):n_min - max(0, -k)])[0, 1]
    for k in lags
])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags, ccf, width=0.8, color='steelblue', alpha=0.8)
ci = 1.96 / np.sqrt(n_min)
ax.axhline(ci,  color='r', ls='--', lw=1, label='±1.96/√n')
ax.axhline(-ci, color='r', ls='--', lw=1)
ax.axvline(0, color='k', lw=0.8, ls=':')
ax.set_xlabel('Lag (hours, temperature leads →)')
ax.set_ylabel('Cross-correlation')
ax.set_title('CCF: temperature increment vs log-price increment')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'increment_ccf.png', dpi=150)
plt.show()

lag0_ccf = float(np.corrcoef(dL_t_n, dL_p_n)[0, 1])
print(f'Cross-correlation at lag 0: {lag0_ccf:.4f}')
print(f'95% CI threshold (±1.96/√n): ±{ci:.4f}')
if abs(lag0_ccf) > ci:
    print('Coupling is statistically significant at lag 0 → Γ ≠ 0')
else:
    print('No significant coupling at lag 0')

## 4.  Save increments to CSV (mode="w", CE-3 fix)

In [ ]:
INCR_DIR.mkdir(parents=True, exist_ok=True)

temp_inc_path  = INCR_DIR / 'temp_inc.csv'
price_inc_path = INCR_DIR / 'price_inc.csv'

# CE-3 fix: mode="w" prevents duplicate rows on re-run
pd.DataFrame({'dL_temp':  dL_temp}).to_csv(temp_inc_path,  mode='w', index=False)
pd.DataFrame({'dL_price': dL_price}).to_csv(price_inc_path, mode='w', index=False)

print(f'Saved {len(dL_temp):,} temperature increments  → {temp_inc_path}')
print(f'Saved {len(dL_price):,} log-price increments   → {price_inc_path}')

# Sanity check: reload and verify shape
t_check = pd.read_csv(temp_inc_path)
p_check = pd.read_csv(price_inc_path)
assert len(t_check) == len(dL_temp),  f'Round-trip mismatch temp: {len(t_check)} != {len(dL_temp)}'
assert len(p_check) == len(dL_price), f'Round-trip mismatch price: {len(p_check)} != {len(dL_price)}'
print('Round-trip check: PASSED')